# eve-api Tutorial

A quick walkthrough of the `eve-api` package: a minimal authenticated HTTP client for the EVE API.

## Installation

```bash
cd eve-api
pip install -e .
```

## 1. Connect and log in

In [ ]:
from eve_api import EVEClient

eve = EVEClient()          # defaults to https://api.eve-chat.chat
await eve.__aenter__()     # open the HTTP connection

await eve.login("your-email@example.com", "your-password")
print("Authenticated:", eve.is_authenticated())

## 2. GET requests

All convenience methods (`get`, `post`, `patch`, `delete`) return parsed JSON as plain Python dicts/lists.

In [ ]:
me = await eve.get("/users/me")
print(f"User: {me['email']}")
print(f"Name: {me.get('first_name', '')} {me.get('last_name', '')}")

In [ ]:
collections = await eve.get("/collections/public")

for col in collections["data"]:
    print(f"  - {col['name']}")

print(f"\nTotal: {collections['meta']['total_count']}")

## 3. POST, PATCH, DELETE

In [ ]:
# Create a conversation
conv = await eve.post("/conversations", json={"name": "Tutorial Chat"})
conv_id = conv["id"]
print(f"Created: {conv_id} — {conv['name']}")

In [ ]:
# Rename it
updated = await eve.patch(f"/conversations/{conv_id}", json={"name": "Renamed Chat"})
print(f"Renamed to: {updated['name']}")

In [ ]:
# Delete it
await eve.delete(f"/conversations/{conv_id}")
print("Deleted.")

## 4. Streaming a response

The `stream()` method yields parsed SSE event dicts as they arrive from the server.

In [ ]:
conv = await eve.post("/conversations", json={"name": "Stream Demo"})
conv_id = conv["id"]

async for event in eve.stream(
    f"/conversations/{conv_id}/stream_messages",
    json={
        "query": "What is Synthetic Aperture Radar?",
        "public_collections": ["eve-public"],
        "k": 3,
    },
):
    if event["type"] == "token":
        print(event["content"], end="", flush=True)
    elif event["type"] == "status":
        print(f"[{event['content']}]", flush=True)
    elif event["type"] == "final":
        print()  # newline after streaming completes

In [ ]:
# Clean up
await eve.delete(f"/conversations/{conv_id}")
print("Conversation deleted.")

## 5. Unauthenticated requests

Use `request()` with `auth_required=False` for endpoints that do not need a token. This returns the raw `httpx.Response`.

In [ ]:
resp = await eve.request("GET", "/health", auth_required=False)
print(f"Status: {resp.status_code}")
print(resp.json())

## 6. Direct token access

Useful if you need to pass the token to another library or service.

In [ ]:
print("Token:", eve.token[:20], "...")
print("Headers:", eve.auth_headers)

## 7. Error handling

The client raises typed exceptions for common HTTP error codes.

In [ ]:
from eve_api import NotFoundError, ForbiddenError, APIError

try:
    await eve.get("/conversations/nonexistent-id")
except NotFoundError:
    print("Not found (404)")
except ForbiddenError:
    print("Forbidden (403)")
except APIError as e:
    print(f"API error {e.status_code}: {e.message}")

## Clean up

In [ ]:
await eve.close()
print("Connection closed.")